# Interactive Exploration: `GDASites` API Tour

This notebook exercises `GDASites` operations:
1. Dynamic path anchoring to workspace root.
2. Loading canonical fixtures vs. production registries.
3. Exact, wildcard, and fuzzy queries.
4. Alias lifecycle management (`add_alias`, `remove_alias`, search resolution).
5. Stateful filtering and batch updates.
6. Testing atomic mutations with isolated scratch persistence in `gtemp/`.

In [ ]:
from pathlib import Path
import os
import sys

# Ensure kernel working directory is anchored to the archive root
cwd = Path.cwd().resolve()
if cwd.name in ("exploration", "maintenance", "notebooks"):
    repo_root = cwd.parents[1] if cwd.name in ("exploration", "maintenance") else cwd.parent
    os.chdir(repo_root)
else:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Workspace Root: {Path.cwd()}")

In [ ]:
from tools.lib.gda_core.GDASites import GDASites
from tools.lib.gda_core.GDAConfig import CONFIG

# Initialize GDASites using the golden test fixture to keep production untouched
golden_fixture = Path("tests/fixtures/sites/golden/golden_sites.json")
if not golden_fixture.exists():
    golden_fixture = Path("data/entities/sites.json")

sites = GDASites(filepath=golden_fixture, auto_load=True)

print(f"Total Sites Loaded: {sites.count}")
print(f"Registry is dirty:  {sites.is_dirty}")

In [ ]:
# 1. Primary Key Lookup & Alias Inspection
sample = sites.get_by_id("SITE-00001")
print(f"Target Site: [{sample['site_id']}] {sample['name']}")
print(f"Existing Aliases: {sample.get('aliases', [])}")

In [ ]:
# 2. Wildcard Search Across Names and Aliases
matches = sites.search(query="*Hospital*", use_fuzzy=False)
print(f"Found {len(matches)} matches for '*Hospital*':")
for m in matches[:5]:
    print(f"  [{m['site_id']}] {m['name']} (Aliases: {m.get('aliases')})")

In [ ]:
# 3. Fuzzy Search with Typo Tolerance Matching Against Aliases
# Typo query for Carlisle Hospital alias
fuzzy_matches = sites.search(query="Carlisle Hospitla", use_fuzzy=True, fuzzy_threshold=0.8)
for m in fuzzy_matches:
    print(f"Fuzzy Match: [{m['site_id']}] {m['name']} | Aliases: {m.get('aliases')}")

In [ ]:
# 4. Dynamic Alias Management: Adding and Verifying Aliases
target_id = "SITE-00001"

# Add a new historical variant alias
added = sites.add_alias(target_id, "Old Carlisle Infirmary")
print(f"Added 'Old Carlisle Infirmary': {added}")
print(f"Registry is dirty: {sites.is_dirty}")

# Verify search resolves immediately via new alias
alias_search = sites.search(query="Old Carlisle Infirmary", use_fuzzy=False)
print(f"Search for 'Old Carlisle Infirmary' resolved: [{alias_search[0]['site_id']}] {alias_search[0]['name']}")

# Duplicate alias attempt (should return False)
dup_added = sites.add_alias(target_id, "Old Carlisle Infirmary")
print(f"Duplicate addition rejected: {dup_added is False}")

In [ ]:
# 5. Dynamic Alias Management: Pruning Aliases
removed = sites.remove_alias(target_id, "Old Carlisle Infirmary")
print(f"Removed 'Old Carlisle Infirmary': {removed}")

# Verify the alias is gone
updated_aliases = sites.get_by_id(target_id)["aliases"]
print(f"Active Aliases for {target_id}: {updated_aliases}")

In [ ]:
# 6. Stateful View Filtering
cemetery_count = sites.set_filter(site_type="CEMETERY")
print(f"Active Cemeteries Filter: {cemetery_count} matches")

# Inspect filtered set
filtered = sites.get_filtered()
for c in filtered[:5]:
    print(f"  [{c['site_id']}] {c['name']} -> {c['location_id']}")

# Clear the filter
sites.clear_filter()
print(f"Filter cleared. Active set restored to {sites.filtered_count} records.")

In [ ]:
# 7. Adding a Site with Auto-Minting and Safe Scratch Persistence
new_id = sites.add(
    site_type="CHURCH",
    name="First Evangelical Meeting House",
    location_id="LOC-00006",
    aliases=["Old Meeting House", "Meeting House on the Square"]
)
print(f"Minted New ID: {new_id}")
print(f"Registry is dirty: {sites.is_dirty}")

# Persist exclusively to gtemp scratch space without altering original files
scratch_target = Path("gtemp/sites_notebook_scratch.json")
sites.save(output_path=scratch_target, force=True)
print(f"Safely written to scratch target: {scratch_target.resolve()}")